# IRIS: three-input training in Google Colab
Run cells in order with a **CPU / Python 3.12** runtime. No GPU or GitHub token is needed.

Download the private repository using **Code > Download ZIP**, then upload that ZIP when requested below. The original `IRIS_Model_Code_v2.zip` also works. Use only synthetic or explicitly authorized data.

This notebook trains the ML component. It does not deploy the older backend, start live connectors, execute controls or update an official risk register. Synthetic models remain **shadow/test only**.

In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Import files from google.colab to provide the Colab browser upload/download interface.
from google.colab import files
# IRIS EXPLAIN: Import Path from pathlib to resolve server-owned paths without trusting uploaded filenames.
from pathlib import Path
# IRIS EXPLAIN: Import json to encode and decode structured finite JSON payloads.
import json
# IRIS EXPLAIN: Import shutil to copy immutable inputs or clean test-owned temporary directories.
import shutil
# IRIS EXPLAIN: Import subprocess to launch explicit notebook/test commands in fresh Python processes.
import subprocess
# IRIS EXPLAIN: Import sys to inspect the Python version or select the same executable/import path.
import sys
# IRIS EXPLAIN: Import tempfile to create fresh isolated run/upload/test directories.
import tempfile
# IRIS EXPLAIN: Import zipfile to inspect/package ZIP contents with explicit source-path checks.
import zipfile

# IRIS EXPLAIN: Call print('Python:', sys.version) using the existing explicit arguments.
print('Python:', sys.version)
# IRIS EXPLAIN: Enter this branch only when sys.version_info[:2] != (3, 12).
if sys.version_info[:2] != (3, 12):
    # IRIS EXPLAIN: Stop this invalid/unauthorized/stale path by raising RuntimeError('Select a Python 3.12 runtime under
    # IRIS EXPLAIN: Runtime > Change runtime type, then rerun this cell.'); the caller's existing boundary determines rollback
    # IRIS EXPLAIN: or the HTTP diagnostic.
    raise RuntimeError('Select a Python 3.12 runtime under Runtime > Change runtime type, then rerun this cell.')
# IRIS EXPLAIN: Store the result of files.upload() in uploaded; later lines use this named result.
uploaded = files.upload()
# IRIS EXPLAIN: Store the comprehension [Path(name) for name in uploaded if name.lower().endswith('.zip')] in archives; later
# IRIS EXPLAIN: lines use this named result.
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
# IRIS EXPLAIN: Enter this branch only when len(archives) != 1.
if len(archives) != 1:
    # IRIS EXPLAIN: Stop this invalid/unauthorized/stale path by raising ValueError('Upload exactly one IRIS repository or
    # IRIS EXPLAIN: model-package ZIP.'); the caller's existing boundary determines rollback or the HTTP diagnostic.
    raise ValueError('Upload exactly one IRIS repository or model-package ZIP.')
# IRIS EXPLAIN: Store the result of Path(tempfile.mkdtemp(prefix='iris_project_', dir='/content')).resolve() (normalize the
# IRIS EXPLAIN: filesystem path) in WORK; later lines use this named result.
WORK = Path(tempfile.mkdtemp(prefix='iris_project_', dir='/content')).resolve()
# IRIS EXPLAIN: Enter zipfile.ZipFile(archives[0]) as archive; its context manager supplies the existing
# IRIS EXPLAIN: cleanup/transaction/expected-exception behavior.
with zipfile.ZipFile(archives[0]) as archive:
    # IRIS EXPLAIN: Iterate archive.infolist(), binding each item to member; process only the existing observed/fixture
    # IRIS EXPLAIN: collection.
    for member in archive.infolist():
        # IRIS EXPLAIN: Enter this branch only when not (WORK / member.filename).resolve().is_relative_to(WORK).
        if not (WORK / member.filename).resolve().is_relative_to(WORK):
            # IRIS EXPLAIN: Stop this invalid/unauthorized/stale path by raising ValueError('Unsafe path inside ZIP.'); the
            # IRIS EXPLAIN: caller's existing boundary determines rollback or the HTTP diagnostic.
            raise ValueError('Unsafe path inside ZIP.')
    # IRIS EXPLAIN: Call archive.extractall(WORK) using the existing explicit arguments.
    archive.extractall(WORK)
# IRIS EXPLAIN: Store the comprehension [p.parent for p in WORK.rglob('requirements.txt') if (p.parent / 'iris_models' /
# IRIS EXPLAIN: '__init__.py').is_file()] in candidates; later lines use this named result.
candidates = [p.parent for p in WORK.rglob('requirements.txt')
              # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
              # IRIS EXPLAIN: if (p.parent / 'iris_models' / '__init__.py').is_file()].
              if (p.parent / 'iris_models' / '__init__.py').is_file()]
# IRIS EXPLAIN: Enter this branch only when len(candidates) != 1.
if len(candidates) != 1:
    # IRIS EXPLAIN: Stop this invalid/unauthorized/stale path by raising ValueError('Could not find exactly one iris_models
    # IRIS EXPLAIN: project in the ZIP.'); the caller's existing boundary determines rollback or the HTTP diagnostic.
    raise ValueError('Could not find exactly one iris_models project in the ZIP.')
# IRIS EXPLAIN: Store candidates[0] in PROJECT; later lines use this named result.
PROJECT = candidates[0]
# IRIS EXPLAIN: Store the computed value PROJECT / 'samples' / 'three_inputs' in SAMPLES; later lines use this named result.
SAMPLES = PROJECT / 'samples' / 'three_inputs'
# IRIS EXPLAIN: Store the result of Path(tempfile.mkdtemp(prefix='iris_run_', dir='/content')) in RUN_ROOT; later lines use this
# IRIS EXPLAIN: named result.
RUN_ROOT = Path(tempfile.mkdtemp(prefix='iris_run_', dir='/content'))
# IRIS EXPLAIN: Store the computed value RUN_ROOT / 'training' in TRAINING; later lines use this named result.
TRAINING = RUN_ROOT / 'training'
# IRIS EXPLAIN: Store the computed value RUN_ROOT / 'prediction_preview' in PREDICTIONS; later lines use this named result.
PREDICTIONS = RUN_ROOT / 'prediction_preview'
# IRIS EXPLAIN: Call print('Project:', PROJECT) using the existing explicit arguments.
print('Project:', PROJECT)
# IRIS EXPLAIN: Call print('Fresh run directory:', RUN_ROOT) using the existing explicit arguments.
print('Fresh run directory:', RUN_ROOT)


## Install the exact dependencies and run the tests
The CLI runs in fresh Python subprocesses, so it does not reuse packages already imported into the notebook kernel. Keep the same pinned dependency versions on the eventual backend.

In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Call subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')],
# IRIS EXPLAIN: check=True) using the existing explicit arguments.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
# IRIS EXPLAIN: Call subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-q'], cwd=PROJECT,
# IRIS EXPLAIN: check=True) using the existing explicit arguments.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-q'], cwd=PROJECT, check=True)


## Combine the sources and train both model components
The sources are asset inventory, risk register and SIEM alerts. `training_labels.jsonl` provides **offline answers only**. It is not a fourth runtime input.

The supplied sources are synthetic; keep `--synthetic`. For real data, prepare sessions, obtain independent session/control reviews, use real provenance, and omit that flag. Do not present generated labels or synthetic scores as real-world validation.

In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Function iris: Run the existing iris_models CLI in a fresh Python process rooted at the uploaded Colab project,
# IRIS EXPLAIN: raising on command failure.
# IRIS EXPLAIN: Inputs: arguments: explicit CLI argument values forwarded to the same Python executable.
def iris(*arguments):
    # IRIS EXPLAIN: Return the result of subprocess.run([sys.executable, '-m', 'iris_models', *map(str, arguments)],
    # IRIS EXPLAIN: cwd=PROJECT, check=True) to the caller; this ends the current function path.
    return subprocess.run([sys.executable, '-m', 'iris_models', *map(str, arguments)], cwd=PROJECT, check=True)

# IRIS EXPLAIN: Call iris('train-three', '--assets', SAMPLES / 'asset_inventory.json', '--risks', SAMPLES /
# IRIS EXPLAIN: 'risk_register.json', '--alerts', SAMPLES / 'siem_alerts.jsonl', '--la... using the existing explicit arguments.
iris('train-three',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--assets', SAMPLES / 'asset_inventory.json',.
     '--assets', SAMPLES / 'asset_inventory.json',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--risks', SAMPLES / 'risk_register.json',.
     '--risks', SAMPLES / 'risk_register.json',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--alerts', SAMPLES / 'siem_alerts.jsonl',.
     '--alerts', SAMPLES / 'siem_alerts.jsonl',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--labels', SAMPLES / 'training_labels.jsonl',.
     '--labels', SAMPLES / 'training_labels.jsonl',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--synthetic', '--out', TRAINING).
     '--synthetic', '--out', TRAINING)


In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Store the result of json.loads((TRAINING / 'models' / 'metadata.json').read_text()) (decode the existing JSON
# IRIS EXPLAIN: text) in metadata; later lines use this named result.
metadata = json.loads((TRAINING / 'models' / 'metadata.json').read_text())
# IRIS EXPLAIN: Call print(json.dumps({'selected_scenario_algorithm': metadata['model1']['algorithm'], 'model1_test_metrics':
# IRIS EXPLAIN: metadata['model1']['metrics'], 'model1_release_allow... using the existing explicit arguments.
print(json.dumps({
    # IRIS EXPLAIN: Continue the current record with selected_scenario_algorithm; each value comes from the expression shown on
    # IRIS EXPLAIN: this unchanged line.
    'selected_scenario_algorithm': metadata['model1']['algorithm'],
    # IRIS EXPLAIN: Continue the current record with model1_test_metrics; each value comes from the expression shown on this
    # IRIS EXPLAIN: unchanged line.
    'model1_test_metrics': metadata['model1']['metrics'],
    # IRIS EXPLAIN: Continue the current record with model1_release_allowed; each value comes from the expression shown on this
    # IRIS EXPLAIN: unchanged line.
    'model1_release_allowed': metadata['model1']['eligible_for_reviewed_advisory_release'],
    # IRIS EXPLAIN: Continue the current record with model2_status; each value comes from the expression shown on this unchanged
    # IRIS EXPLAIN: line.
    'model2_status': metadata['model2']['status'],
    # IRIS EXPLAIN: Continue the current record with model2_release_allowed; each value comes from the expression shown on this
    # IRIS EXPLAIN: unchanged line.
    'model2_release_allowed': metadata['model2']['eligible_for_reviewed_advisory_release']
# IRIS EXPLAIN: Finish the existing grouped expression/call or collection; any trailing fields/call chain remain unchanged.
}, indent=2))
# IRIS EXPLAIN: Call print('Synthetic software check only. Failed or synthetic release gates must not be bypassed.') using the
# IRIS EXPLAIN: existing explicit arguments.
print('Synthetic software check only. Failed or synthetic release gates must not be bypassed.')


## Predict using only the three operational inputs
This makes a **five-session preview**, not a full batch or a new accuracy test. Accuracy is assessed on the held-out split in the evaluation report, not by re-predicting the supplied training data. Remove `--limit 5` below for a full batch.

In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Call iris('predict-three', '--assets', SAMPLES / 'asset_inventory.json', '--risks', SAMPLES /
# IRIS EXPLAIN: 'risk_register.json', '--alerts', SAMPLES / 'siem_alerts.jsonl', '--... using the existing explicit arguments.
iris('predict-three',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--assets', SAMPLES / 'asset_inventory.json',.
     '--assets', SAMPLES / 'asset_inventory.json',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--risks', SAMPLES / 'risk_register.json',.
     '--risks', SAMPLES / 'risk_register.json',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--alerts', SAMPLES / 'siem_alerts.jsonl',.
     '--alerts', SAMPLES / 'siem_alerts.jsonl',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items:
     # IRIS EXPLAIN: '--models', TRAINING / 'models',.
     '--models', TRAINING / 'models',
     # IRIS EXPLAIN: Continue the surrounding call, schema/list, or comprehension with these existing arguments/items: '--mode',
     # IRIS EXPLAIN: 'shadow', '--limit', '5', '--out', PREDICTIONS).
     '--mode', 'shadow', '--limit', '5', '--out', PREDICTIONS)
# IRIS EXPLAIN: Store the result of json.loads((PREDICTIONS / 'prediction_report.json').read_text()) (decode the existing JSON
# IRIS EXPLAIN: text) in report; later lines use this named result.
report = json.loads((PREDICTIONS / 'prediction_report.json').read_text())
# IRIS EXPLAIN: Assert report['training_labels_read'] is False; a failure indicates the demonstrated invariant did not hold.
assert report['training_labels_read'] is False
# IRIS EXPLAIN: Call print({key: report[key] for key in ('total_sessions', 'predictions_written', 'limited_preview',
# IRIS EXPLAIN: 'training_labels_read', 'automatic_execution')}) using the existing explicit arguments.
print({key: report[key] for key in ('total_sessions', 'predictions_written', 'limited_preview', 'training_labels_read', 'automatic_execution')})
# IRIS EXPLAIN: Enter (PREDICTIONS / 'predictions.jsonl').open() as preview; its context manager supplies the existing
# IRIS EXPLAIN: cleanup/transaction/expected-exception behavior.
with (PREDICTIONS / 'predictions.jsonl').open() as preview:
    # IRIS EXPLAIN: Call print(json.dumps(json.loads(next(preview)), indent=2)) using the existing explicit arguments.
    print(json.dumps(json.loads(next(preview)), indent=2))


## Download the complete run
Colab storage is temporary. Download the outputs now. Keep the **entire models directory**, reports, original code, matching control catalogue and pinned requirements. Load only artifacts you trained or independently trust: `joblib`/pickle can execute code.

The next engineering step is a backend adapter using the same three-source preparation and session contract. This notebook does not wire that adapter or deploy a website.

In [ ]:
# IRIS EXPLAIN: File purpose: Colab code cell: use the existing uploaded project and pinned environment; execution order and
# IRIS EXPLAIN: commands are unchanged.
# IRIS EXPLAIN: Store the result of shutil.make_archive(str(RUN_ROOT) + '_results', 'zip', root_dir=RUN_ROOT) in export_path;
# IRIS EXPLAIN: later lines use this named result.
export_path = shutil.make_archive(str(RUN_ROOT) + '_results', 'zip', root_dir=RUN_ROOT)
# IRIS EXPLAIN: Call files.download(export_path) using the existing explicit arguments.
files.download(export_path)
